# 01 — Enterprise search pipeline architecture

> **Applied Labs** · **Domain**: RAG · **Difficulty**: Advanced · **Reading time**: ~35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/12_enterprise_search/01_architecture.ipynb)

**Prerequisites**:
- [00_first_principles.ipynb](./00_first_principles.ipynb) — BM25 failures, vocabulary mismatch, market context

**What you will learn**:
- End-to-end architecture for enterprise search: connectors → answer generation
- Connector design patterns for heterogeneous data sources
- Four chunking strategies and when to use each
- Hybrid retrieval: dense + sparse + reciprocal rank fusion
- Prompt engineering for grounded answers with verifiable citations
- Access control architecture: per-doc ACLs and query-time filtering

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "numpy>=1.24.0"

import os, re, json, textwrap, math, hashlib
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set, Protocol
from abc import ABC, abstractmethod
from collections import Counter, defaultdict

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — System architecture

The enterprise search pipeline has **7 stages**, each with distinct responsibilities:

```
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│Connectors│──▶│ Chunker  │──▶│ Embedder │──▶│  Index   │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
                                                   │
┌──────────┐   ┌──────────┐   ┌──────────┐        │
│ Response  │◀──│Generator │◀──│ Reranker │◀───────┘
│ + Cites  │   │  (LLM)   │   │          │◀── Query
└──────────┘   └──────────┘   └──────────┘
```

**Ingestion path** (offline): Connectors → Chunker → Embedder → Index
**Query path** (online): Query → Retriever → Reranker → Generator → Citations

Let's define each component's interface.

In [ ]:
@dataclass
class RawDocument:
    """A document as ingested from a source connector."""
    doc_id: str
    title: str
    content: str
    source_type: str       # wiki, chat, docs, email
    metadata: Dict = field(default_factory=dict)
    acl_groups: Set[str] = field(default_factory=lambda: {"all_employees"})
    updated_at: str = ""   # ISO timestamp

@dataclass
class Chunk:
    """A chunk produced by the chunking pipeline."""
    chunk_id: str
    doc_id: str
    text: str
    index: int             # position within the parent document
    metadata: Dict = field(default_factory=dict)

@dataclass
class ScoredChunk:
    """A chunk with a retrieval score."""
    chunk: Chunk
    score: float
    method: str            # bm25, dense, hybrid

@dataclass
class SearchResult:
    """Final answer with source citations."""
    query: str
    answer: str
    citations: List[Dict]  # [{chunk_id, doc_id, title, snippet, score}]
    confidence: float


# Pipeline stage interfaces
class Connector(ABC):
    @abstractmethod
    def fetch(self) -> List[RawDocument]: ...

class Chunker(ABC):
    @abstractmethod
    def chunk(self, doc: RawDocument) -> List[Chunk]: ...

class Retriever(ABC):
    @abstractmethod
    def retrieve(self, query: str, top_k: int = 5) -> List[ScoredChunk]: ...

class Generator(ABC):
    @abstractmethod
    def generate(self, query: str, chunks: List[ScoredChunk]) -> SearchResult: ...


print("Pipeline data models defined ✓")
print(f"  RawDocument → Chunk → ScoredChunk → SearchResult")
print(f"  Stages: Connector → Chunker → Retriever → Generator")

## Section 2 — Connector design

Enterprise data lives in **heterogeneous systems** — each with its own API,
authentication, and data model. A **connector** abstracts this complexity behind
a uniform `RawDocument` interface.

Key design decisions:
- **Incremental sync** — Only fetch documents changed since last sync
- **Metadata preservation** — Author, timestamps, permissions, tags
- **Format normalization** — HTML → plain text, PDF → text, etc.

In [ ]:
class WikiConnector(Connector):
    """Connector for internal wiki (e.g., Confluence, Notion)."""

    def __init__(self, pages: List[Dict]):
        self.pages = pages

    def fetch(self) -> List[RawDocument]:
        docs: List[RawDocument] = []
        for page in self.pages:
            doc = RawDocument(
                doc_id=f"wiki-{page['id']}",
                title=page["title"],
                content=self._strip_html(page.get("html", page.get("text", ""))),
                source_type="wiki",
                metadata={"author": page.get("author", "unknown"),
                          "space": page.get("space", "general")},
                acl_groups=set(page.get("acl", ["all_employees"])),
                updated_at=page.get("updated", ""),
            )
            docs.append(doc)
        return docs

    @staticmethod
    def _strip_html(html: str) -> str:
        return re.sub(r"<[^>]+>", "", html)


class ChatConnector(Connector):
    """Connector for chat platforms (e.g., Slack, Teams)."""

    def __init__(self, messages: List[Dict]):
        self.messages = messages

    def fetch(self) -> List[RawDocument]:
        # Group messages by thread
        threads: Dict[str, List[Dict]] = defaultdict(list)
        for msg in self.messages:
            threads[msg.get("thread_id", msg["id"])].append(msg)

        docs: List[RawDocument] = []
        for thread_id, msgs in threads.items():
            combined = "\n".join(f"@{m['author']}: {m['text']}" for m in msgs)
            doc = RawDocument(
                doc_id=f"chat-{thread_id}",
                title=f"Chat thread: {msgs[0]['text'][:50]}...",
                content=combined,
                source_type="chat",
                metadata={"channel": msgs[0].get("channel", "general"),
                          "participants": list({m["author"] for m in msgs})},
                updated_at=msgs[-1].get("timestamp", ""),
            )
            docs.append(doc)
        return docs


class EmailConnector(Connector):
    """Connector for email archives (e.g., Gmail, Outlook)."""

    def __init__(self, emails: List[Dict]):
        self.emails = emails

    def fetch(self) -> List[RawDocument]:
        docs: List[RawDocument] = []
        for email in self.emails:
            content = f"From: {email['from']}\nTo: {email['to']}\n"
            content += f"Subject: {email['subject']}\n\n{email['body']}"
            doc = RawDocument(
                doc_id=f"email-{email['id']}",
                title=email["subject"],
                content=content,
                source_type="email",
                metadata={"from": email["from"], "to": email["to"]},
                acl_groups=set(email.get("acl", [email["to"]])),
                updated_at=email.get("date", ""),
            )
            docs.append(doc)
        return docs


# Demonstrate connector usage
wiki_data = [
    {"id": "101", "title": "PTO Policy", "text": "Employees earn 15 days PTO per year.",
     "author": "hr_admin", "space": "HR", "acl": ["all_employees"]},
]
chat_data = [
    {"id": "m1", "thread_id": "t1", "author": "alice", "text": "How do I request PTO?",
     "channel": "hr-questions", "timestamp": "2024-06-01T10:00:00"},
    {"id": "m2", "thread_id": "t1", "author": "bob", "text": "Submit via HR portal, 2 weeks ahead.",
     "channel": "hr-questions", "timestamp": "2024-06-01T10:05:00"},
]
email_data = [
    {"id": "e1", "from": "cfo@company.com", "to": "finance_team",
     "subject": "Q3 Budget Review", "body": "Please review attached budget spreadsheet.",
     "acl": ["finance_team", "exec_team"], "date": "2024-07-15"},
]

wiki_conn = WikiConnector(wiki_data)
chat_conn = ChatConnector(chat_data)
email_conn = EmailConnector(email_data)

all_docs: List[RawDocument] = []
for conn in [wiki_conn, chat_conn, email_conn]:
    fetched = conn.fetch()
    all_docs.extend(fetched)
    print(f"{conn.__class__.__name__}: fetched {len(fetched)} document(s)")

print(f"\nTotal documents ingested: {len(all_docs)}")
for doc in all_docs:
    print(f"  [{doc.source_type}] {doc.doc_id}: {doc.title}")
    print(f"    ACL: {doc.acl_groups}")

## Section 3 — Chunking strategies

Documents must be split into **chunks** for embedding and retrieval. The chunking
strategy significantly affects retrieval quality. We compare four approaches:

| Strategy | How it splits | Best for |
|----------|--------------|----------|
| **Sentence** | One chunk = one sentence | Short, factual docs |
| **Paragraph** | Split on double newlines | Well-structured articles |
| **Fixed-window** | N tokens with overlap | Uniform chunk sizes |
| **Semantic** | Split when topic shifts | Long, multi-topic docs |

In [ ]:
SAMPLE_DOC = """Employee Benefits Overview

Health Insurance: All full-time employees are eligible for health insurance starting day one.
We offer three plans: Basic, Standard, and Premium. Dependents can be added during open enrollment.

Retirement: The company matches 401(k) contributions up to 4% of salary. Vesting is immediate.
Employees can adjust contribution levels quarterly through the benefits portal.

Paid Time Off: Full-time employees earn 15 days PTO per year. PTO accrues monthly.
Unused days roll over up to 5 days. Requests require 2-week advance notice via HR portal.

Professional Development: Each employee has a $2,000 annual learning budget.
Approved expenses include conferences, courses, books, and certifications.
Manager approval is required for expenses over $500."""


class SentenceChunker(Chunker):
    """Split on sentence boundaries."""
    def chunk(self, doc: RawDocument) -> List[Chunk]:
        sentences = re.split(r'(?<=[.!?])\s+', doc.content.strip())
        return [Chunk(f"{doc.doc_id}-s{i}", doc.doc_id, s.strip(), i)
                for i, s in enumerate(sentences) if s.strip()]


class ParagraphChunker(Chunker):
    """Split on double newlines (paragraphs)."""
    def chunk(self, doc: RawDocument) -> List[Chunk]:
        paragraphs = re.split(r'\n\n+', doc.content.strip())
        return [Chunk(f"{doc.doc_id}-p{i}", doc.doc_id, p.strip(), i)
                for i, p in enumerate(paragraphs) if p.strip()]


class FixedWindowChunker(Chunker):
    """Fixed token window with overlap."""
    def __init__(self, window: int = 50, overlap: int = 10):
        self.window = window
        self.overlap = overlap

    def chunk(self, doc: RawDocument) -> List[Chunk]:
        words = doc.content.split()
        chunks: List[Chunk] = []
        start = 0
        idx = 0
        while start < len(words):
            end = min(start + self.window, len(words))
            text = " ".join(words[start:end])
            chunks.append(Chunk(f"{doc.doc_id}-w{idx}", doc.doc_id, text, idx))
            start += self.window - self.overlap
            idx += 1
        return chunks


class SemanticChunker(Chunker):
    """Split when topic shifts (detected by heading or keyword change)."""
    def chunk(self, doc: RawDocument) -> List[Chunk]:
        # Heuristic: split on lines that look like section headers
        sections: List[str] = []
        current: List[str] = []
        for line in doc.content.split("\n"):
            stripped = line.strip()
            if stripped and stripped.endswith(":") and len(stripped.split()) <= 5:
                if current:
                    sections.append("\n".join(current))
                current = [stripped]
            else:
                current.append(line)
        if current:
            sections.append("\n".join(current))
        return [Chunk(f"{doc.doc_id}-sem{i}", doc.doc_id, s.strip(), i)
                for i, s in enumerate(sections) if s.strip()]


# Compare all four strategies
sample_raw = RawDocument("benefits", "Employee Benefits", SAMPLE_DOC, "wiki")
chunkers = {
    "Sentence":     SentenceChunker(),
    "Paragraph":    ParagraphChunker(),
    "Fixed(50,10)": FixedWindowChunker(50, 10),
    "Semantic":     SemanticChunker(),
}

print(f"{'Strategy':<16} {'Chunks':>6}  {'Avg len (chars)':>16}  {'Min':>5}  {'Max':>5}")
print("-" * 60)
for name, chunker in chunkers.items():
    chunks = chunker.chunk(sample_raw)
    lengths = [len(c.text) for c in chunks]
    print(f"{name:<16} {len(chunks):>6}  {np.mean(lengths):>16.0f}  {min(lengths):>5}  {max(lengths):>5}")

# Show semantic chunks in detail
print("\n--- Semantic chunks ---")
sem_chunks = SemanticChunker().chunk(sample_raw)
for c in sem_chunks:
    preview = c.text[:80].replace("\n", " ") + ("..." if len(c.text) > 80 else "")
    print(f"  [{c.chunk_id}] {preview}")

## Section 4 — Hybrid retrieval

Neither dense (embedding) nor sparse (BM25) retrieval alone is optimal:
- **Dense** handles synonyms but struggles with exact keyword matches
- **BM25** excels at exact matches but misses semantic equivalence

**Hybrid retrieval** combines both using **Reciprocal Rank Fusion (RRF)**:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60) and $\text{rank}_r(d)$ is the rank of
document $d$ in ranker $r$'s list.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: List[List[Tuple[str, float]]],
    k: int = 60
) -> List[Tuple[str, float]]:
    """Fuse multiple ranked lists using Reciprocal Rank Fusion.

    Args:
        ranked_lists: Each list contains (doc_id, score) tuples, ordered by score desc.
        k: Smoothing constant (default 60).

    Returns:
        Fused ranking as (doc_id, rrf_score) tuples, ordered by score desc.
    """
    rrf_scores: Dict[str, float] = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked_list, 1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: -x[1])


# Simulated rankings — dense and sparse disagree on order
bm25_ranking = [("D02", 8.5), ("D04", 7.2), ("D10", 6.8), ("D01", 5.1), ("D03", 4.0)]
dense_ranking = [("D01", 0.92), ("D02", 0.88), ("D03", 0.81), ("D10", 0.77), ("D04", 0.72)]

fused = reciprocal_rank_fusion([bm25_ranking, dense_ranking])

print("Query: 'how to handle customer refunds'\n")
print(f"{'Rank':<6} {'BM25':<20} {'Dense':<20} {'Hybrid (RRF)':<20}")
print("-" * 66)
for i in range(5):
    bm25_entry = f"{bm25_ranking[i][0]} ({bm25_ranking[i][1]:.1f})" if i < len(bm25_ranking) else ""
    dense_entry = f"{dense_ranking[i][0]} ({dense_ranking[i][1]:.2f})" if i < len(dense_ranking) else ""
    fused_entry = f"{fused[i][0]} ({fused[i][1]:.4f})" if i < len(fused) else ""
    print(f"{i+1:<6} {bm25_entry:<20} {dense_entry:<20} {fused_entry:<20}")

print("\n→ RRF boosts D01 (correct answer) and D02 (process doc) to top-2.")
print("  Neither BM25 nor dense alone ranked D01 first.")

## Section 5 — Answer generation with citations

The generator takes retrieved chunks and produces a **grounded answer** with
**source citations**. Key design principles:

1. **Faithfulness** — Every claim must be supported by a retrieved chunk
2. **Citation format** — Inline `[1]` references mapped to source documents
3. **Abstention** — Say "I don't have enough information" rather than hallucinate
4. **Conflict resolution** — Flag when sources disagree

In [ ]:
SYSTEM_PROMPT = """You are an enterprise search assistant. Answer the user's question
using ONLY the provided context chunks. Follow these rules strictly:

1. CITE every factual claim with [n] where n is the chunk number.
2. If the context does not contain enough information, say "I don't have enough
   information to answer this question fully" and explain what's missing.
3. If sources conflict, note the conflict and present both versions with citations.
4. Never fabricate information not present in the provided chunks.
5. Keep answers concise — aim for 2-4 sentences unless the question requires detail.
"""


def build_generation_prompt(
    query: str,
    chunks: List[Dict[str, str]],
    system_prompt: str = SYSTEM_PROMPT
) -> List[Dict[str, str]]:
    """Build the chat messages for answer generation.

    Args:
        query: The user's search query.
        chunks: List of {"id": ..., "text": ...} dicts.
        system_prompt: System instructions for grounded generation.

    Returns:
        List of chat messages for the OpenAI API.
    """
    context_block = "\n\n".join(
        f"[{i+1}] (ID: {c['id']})\n{c['text']}" for i, c in enumerate(chunks)
    )
    user_message = f"""Context chunks:
{context_block}

Question: {query}

Answer with citations [n]:"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]


# Example: build prompt for a sample query
sample_chunks = [
    {"id": "D05-p3", "text": "Full-time employees earn 15 days PTO per year. "
     "Unused days roll over up to 5 days."},
    {"id": "chat-t1-s2", "text": "@bob: Submit via HR portal, 2 weeks ahead."},
    {"id": "D08-p1", "text": "Day 1: laptop setup, VPN access. Day 2: codebase walkthrough."},
]

messages = build_generation_prompt("How much PTO do employees get?", sample_chunks)

print("=== GENERATION PROMPT ===\n")
for msg in messages:
    print(f"[{msg['role'].upper()}]")
    print(msg["content"][:500])
    print()

# Expected answer (simulated)
expected_answer = (
    "Full-time employees earn 15 days of PTO per year [1]. Unused days roll over "
    "up to 5 days [1]. PTO requests should be submitted via the HR portal at least "
    "2 weeks in advance [2]."
)
print("=== EXPECTED ANSWER ===")
print(expected_answer)

# Abstention example
print("\n=== ABSTENTION EXAMPLE ===")
print("Q: What is the company's parental leave policy?")
print("A: I don't have enough information to answer this question. The provided")
print("   context covers PTO and onboarding but does not mention parental leave.")

## Section 6 — Access control architecture

Enterprise search **must** enforce document-level permissions at query time.
The architecture has three layers:

1. **Document-level ACLs** — Each document stores which groups can access it
2. **Query-time filtering** — Retriever filters results by user's group membership
3. **Permission inheritance** — Folder/space permissions propagate to child docs

In [ ]:
@dataclass
class User:
    """An enterprise user with group memberships."""
    user_id: str
    name: str
    groups: Set[str]


@dataclass
class ACLEntry:
    """Access control entry for a document."""
    doc_id: str
    allowed_groups: Set[str]
    inherited_from: Optional[str] = None  # parent folder/space ID


class ACLFilteredRetriever:
    """Wraps a retriever with ACL-based filtering."""

    def __init__(self, acl_index: Dict[str, ACLEntry]):
        self.acl_index = acl_index

    def filter_results(
        self, results: List[Dict], user: User
    ) -> List[Dict]:
        """Remove results the user is not authorized to see.

        Args:
            results: Ranked search results with doc_id.
            user: The querying user.

        Returns:
            Filtered results preserving rank order.
        """
        filtered: List[Dict] = []
        blocked: List[str] = []
        for result in results:
            acl = self.acl_index.get(result["doc_id"])
            if acl is None or bool(user.groups & acl.allowed_groups):
                filtered.append(result)
            else:
                blocked.append(result["doc_id"])
        return filtered, blocked


# Build ACL index
acl_index: Dict[str, ACLEntry] = {
    "D01": ACLEntry("D01", {"all_employees"}),
    "D05": ACLEntry("D05", {"all_employees"}),
    "D06": ACLEntry("D06", {"engineering", "devops"}),
    "SAL": ACLEntry("SAL", {"hr_team", "exec_team"}),
    "MA":  ACLEntry("MA",  {"exec_team"}),
    "D11": ACLEntry("D11", {"all_employees", "security_team"}),
}

# Simulate query results (before ACL filtering)
raw_results = [
    {"doc_id": "D01", "title": "Return & Exchange Guidelines", "score": 0.95},
    {"doc_id": "SAL", "title": "2024 Salary Bands",            "score": 0.88},
    {"doc_id": "MA",  "title": "Acquisition Target — Acme",    "score": 0.82},
    {"doc_id": "D06", "title": "Kubernetes Runbook",            "score": 0.75},
    {"doc_id": "D05", "title": "PTO Policy",                    "score": 0.70},
]

# Two users with different permissions
users = [
    User("u1", "Alice (Engineering)", {"all_employees", "engineering"}),
    User("u2", "Bob (Executive)",     {"all_employees", "exec_team", "hr_team"}),
]

retriever = ACLFilteredRetriever(acl_index)

for user in users:
    allowed, blocked = retriever.filter_results(raw_results, user)
    print(f"\n{'═' * 50}")
    print(f"User: {user.name}")
    print(f"Groups: {user.groups}")
    print(f"{'═' * 50}")
    print(f"✓ Allowed ({len(allowed)}):")
    for r in allowed:
        print(f"    {r['doc_id']} — {r['title']} (score: {r['score']:.2f})")
    print(f"✗ Blocked ({len(blocked)}): {blocked}")

## 🏋️ Exercise 1 — Design a Jira/GitHub Issues connector

Implement a `TicketConnector` class following the `Connector` interface. It should:
- Accept a list of ticket dicts with fields: `id`, `title`, `description`, `status`, `assignee`, `labels`
- Return `RawDocument` objects with source_type `"ticket"`
- Store `status` and `labels` in metadata

In [ ]:
# YOUR CODE HERE
# class TicketConnector(Connector):
#     def __init__(self, tickets: List[Dict]):
#         ...
#     def fetch(self) -> List[RawDocument]:
#         ...

## 🏋️ Exercise 2 — Implement weighted RRF

Modify `reciprocal_rank_fusion` to accept per-ranker **weights** (e.g., give
dense retrieval 2× the weight of BM25). Test with the example rankings and
show how weights shift the final ordering.

In [ ]:
# YOUR CODE HERE
# def weighted_rrf(
#     ranked_lists: List[List[Tuple[str, float]]],
#     weights: List[float],
#     k: int = 60
# ) -> List[Tuple[str, float]]:
#     ...

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Enterprise search has **7 pipeline stages**: connectors, chunking, embedding, indexing, retrieval, reranking, generation |
| 2 | **Connectors** abstract heterogeneous sources behind a uniform `RawDocument` interface with metadata and ACLs |
| 3 | **Chunking strategy** (sentence, paragraph, fixed-window, semantic) significantly affects retrieval quality |
| 4 | **Hybrid retrieval** (dense + sparse + RRF) outperforms either method alone by combining semantic understanding with exact matching |
| 5 | **Grounded generation** requires strict citation enforcement, abstention rules, and conflict detection |
| 6 | **ACL filtering** at query time is non-negotiable — permission checks must happen before results reach the user |

## What's Next

In **02_build.ipynb**, we implement the full pipeline end-to-end: synthetic
knowledge base → chunking → embedding → hybrid retrieval → RAG answer generation.

## References

1. Craswell, N. et al. (2020). *Overview of the TREC 2019 Deep Learning Track*. TREC.
2. Wang, L. et al. (2023). *Query2doc: Query Expansion with Large Language Models*. EMNLP 2023.
3. Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020.
4. Elastic (2024). Elasticsearch: Enterprise search. https://www.elastic.co